We're pre-training the BERT base model 

In [190]:
# let's compare the two models to be trained 
from torchinfo import summary 
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, AutoModelForMultipleChoice
from transformers import set_seed 

set_seed(42) 
model_name = 'bert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_name)  
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=9)
model2 = AutoModelForMultipleChoice.from_pretrained(model_name)
print(summary(model.distilbert))
print(summary(model))
print(summary(model2))
# We see that the majority is overlapping. 

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForMultipleChoice were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


AttributeError: 'BertForSequenceClassification' object has no attribute 'distilbert'

In [191]:
# load dataset 
import json
import datasets 
from tqdm import tqdm
def read_questions_answers(file_path):
    with open(file_path, 'r') as file:
        json_data = json.load(file)
    questions_data = json_data['questions']

    correct_count = 0
    incorrect_answers = []

    questions = []
    answers2 = [] 
    correct_answers = []

    with tqdm(total=len(questions_data), desc="Processing Questions") as progress_bar:
        for item in questions_data:
            question = item['question']
            answers = item['answers']
            correct_answer = item['solution']

            questions.append(question)
            answers2.append(answers)
            correct_answers.append(correct_answer)
    return questions, answers2, correct_answers

file_path='data/CyberMetric/CyberMetric-10000-v1.json'
questions, answers, correct_answers = read_questions_answers(file_path)

Processing Questions:   0%|                           | 0/10180 [00:00<?, ?it/s]


In [192]:
import pandas as pd 
from datasets import load_dataset
from datasets import Dataset 

answer_mapping = {'A':0,'B':1,'C':2,'D':3}
def preprocess_function(examples):
    first_sentences = [[context] * 4 for context in examples["questions"]]
    second_sentences = [list(en.values()) for en in examples["answers"]]
    #second_sentences = [list(en.values()) for en in [examples["answers"]]]
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])
    tokenized_examples = tokenizer(first_sentences, second_sentences, truncation=True)
    ans = [answer_mapping[correct] for correct in examples["correct"]]
    tokenized_examples["labels"] = sum([[context]* 4 for context in ans], [])
    return {k: [v[i : i + 4] for i in range(0, len(v), 4)] for k, v in tokenized_examples.items()}

In [193]:
data = {'questions': questions, 'answers':answers, 'correct':correct_answers}
df = pd.DataFrame(data)
dataset = Dataset.from_pandas(df, split='train')
processed = preprocess_function(dataset[:2])
processed = dataset.map(preprocess_function, batched=True)
print(processed)

Map:   0%|          | 0/10180 [00:00<?, ? examples/s]

Dataset({
    features: ['questions', 'answers', 'correct', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 10180
})


In [199]:
import evaluate
from transformers import TrainingArguments, Trainer 
training_args = TrainingArguments(
        output_dir="DistilBERTmc",
        learning_rate=1e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        logging_steps=100,
        # I added these three 
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model='accuracy',
        save_total_limit=1,
    )
accuracy = evaluate.load("accuracy")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [195]:
def compute_metrics(eval_pred): 
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [196]:
from dataclasses import dataclass
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from typing import Optional, Union
import torch


@dataclass
class DataCollatorForMultipleChoice:
    """
    Data collator that will dynamically pad the inputs for multiple choice received.
    """

    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else "labels"
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels)[:,0]
        return batch

In [200]:
from sklearn.model_selection import StratifiedKFold, KFold
import numpy as np
k_folds = KFold(n_splits=5, shuffle=True, random_state=42) 
for fold, (train_idx, val_idx) in enumerate(k_folds.split(processed)): 
    model2 = AutoModelForMultipleChoice.from_pretrained(model_name)
    print(f"Training on fold {fold + 1}")
    train_dataset = processed.select(train_idx.tolist())
    val_dataset = processed.select(val_idx.tolist())
    trainer = Trainer(
        model=model2, 
        args = training_args,
        train_dataset=train_dataset, 
        eval_dataset=val_dataset, 
        tokenizer=tokenizer, 
        data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )
    trainer.train()

    break 

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/local/tmp.3482832/ipykernel_867495/803330672.py:9: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Training on fold 1


Epoch,Training Loss,Validation Loss,Accuracy
1,1.138800,1.135596,0.508350
2,1.027100,1.097720,0.535855
3,0.847300,1.140727,0.545187
4,0.727300,1.178577,0.545678
5,0.650300,1.209911,0.545187


In [201]:
output = trainer.predict(val_dataset)
print(sum(np.argmax(output[0],axis=1) == output[1])/ len(output[1]))
print(np.argmax(output[0],axis=1))
print(output[1])

0.5456777996070727
[3 3 2 ... 0 3 2]
[3 3 3 ... 0 3 2]


In [189]:
output = trainer.predict(val_dataset)
print(sum(np.argmax(output[0],axis=1) == output[1])/ len(output[1]))
print(np.argmax(output[0],axis=1))
print(output[1])


0.5039292730844793
[3 3 2 ... 0 3 0]
[3 3 3 ... 0 3 2]


In [157]:
# test data collator: 
accepted_keys = ["input_ids","attention_mask","labels"]
print(processed)
features = [{k: v for k, v in processed[i].items() if k in accepted_keys} for i in range(10)]
batch = DataCollatorForMultipleChoice(tokenizer)(features)
print(batch)

Dataset({
    features: ['questions', 'answers', 'correct', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2000
})
{'input_ids': tensor([[[  101,  2241,  2006,  ...,  2003, 20419,   102],
         [  101,  2241,  2006,  ...,     0,     0,     0],
         [  101,  2241,  2006,  ...,     0,     0,     0],
         [  101,  2241,  2006,  ...,     0,     0,     0]],

        [[  101,  2029,  1997,  ...,     0,     0,     0],
         [  101,  2029,  1997,  ...,     0,     0,     0],
         [  101,  2029,  1997,  ...,     0,     0,     0],
         [  101,  2029,  1997,  ...,     0,     0,     0]],

        [[  101,  2054,  2003,  ...,     0,     0,     0],
         [  101,  2054,  2003,  ...,     0,     0,     0],
         [  101,  2054,  2003,  ...,     0,     0,     0],
         [  101,  2054,  2003,  ...,     0,     0,     0]],

        ...,

        [[  101,  2054,  3252,  ...,     0,     0,     0],
         [  101,  2054,  3252,  ...,     0,     0,     0],
         [  101,

In [87]:
print([tokenizer.decode(processed["input_ids"][idx][i]) for i in range(4)])

['[CLS] in tcp / ip networking, which protocol is used to hold network addresses and routing information in a packet? [SEP] http [SEP]', '[CLS] in tcp / ip networking, which protocol is used to hold network addresses and routing information in a packet? [SEP] ip [SEP]', '[CLS] in tcp / ip networking, which protocol is used to hold network addresses and routing information in a packet? [SEP] routing information protocol ( rip ) [SEP]', '[CLS] in tcp / ip networking, which protocol is used to hold network addresses and routing information in a packet? [SEP] tcp [SEP]']
